## 🎯 Learning Objectives
* Implement and configure advanced retrieval techniques using LlamaIndex on a real-world corpus.
* Understand the trade-offs and benefits of different advanced RAG patterns such as Sentence Window Retrieval and Hypothetical Document Embedding (HyDE).
* Evaluate the performance of advanced retrieval strategies qualitatively by comparing retrieved contexts and generated answers.
* Develop a structured approach to applying advanced RAG patterns for production-ready QA applications.


## RAG02-L09: Exercise: Implement Advanced Retrieval on a Real Corpus

### Task Description

In this exercise, you will apply your knowledge of advanced RAG patterns to a practical scenario. Your goal is to implement and compare at least two distinct advanced retrieval techniques using LlamaIndex on a provided corpus. This will simulate a real-world task of enhancing a Question-Answering (QA) system's retrieval capabilities.

### Corpus

You will be working with a small, representative corpus of LlamaIndex documentation pages. This corpus mimics the complexity and structure you might encounter in enterprise knowledge bases.

### Requirements

1.  **Corpus Loading and Indexing**: Load the provided LlamaIndex documentation files and create a LlamaIndex `VectorStoreIndex` for baseline retrieval.
2.  **Advanced Retrieval Implementation**: Implement at least two of the following advanced retrieval techniques using LlamaIndex:
    *   **Sentence Window Retrieval**: Focus on precise context for re-ranking.
    *   **Hypothetical Document Embedding (HyDE)**: Improve embedding search by generating hypothetical answers.
    *   **RAG-Fusion**: Combine multiple query transformations for robust retrieval.
    *   **Auto-merging Retrieval**: Handle hierarchical document structures effectively.
    *   **Recursive Retriever**: Navigate complex document graphs.
    
    For this exercise, we recommend focusing on **Sentence Window Retrieval** and **Hypothetical Document Embedding (HyDE)** for their distinct approaches to improving retrieval.
3.  **Query Engine Construction**: For each implemented advanced retrieval technique, construct a LlamaIndex `QueryEngine`.
4.  **Qualitative Evaluation**: Formulate at least three diverse test queries. For each query, execute it against the baseline `QueryEngine` and each of your advanced `QueryEngine`s. Compare the retrieved source nodes (context) and the generated answers. Discuss the observed differences and the potential benefits or drawbacks of each technique for the given queries.
5.  **Code Clarity and Comments**: Ensure your code is well-structured, readable, and thoroughly commented, explaining your design choices and the rationale behind using specific LlamaIndex components.

### Evaluation Criteria

*   **Correctness**: Proper implementation of LlamaIndex components for advanced retrieval.
*   **Completeness**: All requirements are met, including loading, indexing, implementing two techniques, and performing qualitative evaluation.
*   **Clarity of Explanation**: Clear and concise explanations of the chosen techniques, their implementation, and the observed results.
*   **Insightfulness**: Thoughtful comparison of retrieval results, highlighting the strengths and weaknesses of each approach.
*   **Code Quality**: Clean, well-commented, and efficient Python code.


In [ ]:
# Install necessary libraries
%pip install -q llama-index==0.10.30 openai==1.30.0 python-dotenv==1.0.1 nest_asyncio==1.6.0

import os
import nest_asyncio
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.indices.postprocessor import SimilarityPostprocessor
from llama_index.core.schema import NodeRelationship, RelatedNodeInfo

# For HyDE
from llama_index.core.indices.query.query_transform import HypotheticalDocumentEmbedder

# Apply nest_asyncio for Jupyter environments
nest_asyncio.apply()

# Load environment variables from .env file
load_dotenv()

# --- Configuration --- 

# Set your OpenAI API key
# Ensure OPENAI_API_KEY is set in your .env file or as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Initialize LLM and Embedding Model
# Using GPT-4o for generation and text-embedding-3-large for embeddings (2026 ready models)
Settings.llm = OpenAI(model="gpt-4o", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

# --- Corpus Setup --- 

# Create a dummy directory and some markdown files for the corpus
# In a real scenario, you would point SimpleDirectoryReader to your actual data folder.
corpus_dir = "./llama_index_docs"
os.makedirs(corpus_dir, exist_ok=True)

doc_contents = {
    "introduction.md": "LlamaIndex is a data framework for LLM applications. It provides tools to ingest, structure, and access private or domain-specific data. Key features include data connectors, data indexes, and query engines. It supports various data sources like APIs, databases, and PDFs.",
    "query_engines.md": "Query engines in LlamaIndex allow you to ask questions over your data. They take a query string and return a `Response` object. Different types of query engines exist, including `VectorStoreQueryEngine` for semantic search and `SubQuestionQueryEngine` for complex queries. Advanced query engines often incorporate retrieval augmentation techniques.",
    "advanced_rag.md": "Advanced RAG patterns enhance the quality of retrieved context. Techniques include Sentence Window Retrieval, which retrieves a small window and then expands it for the LLM. Another technique is Hypothetical Document Embedding (HyDE), where the LLM first generates a hypothetical answer, which is then embedded and used to retrieve more relevant documents. Auto-merging retrieval focuses on hierarchical document structures.",
    "indexing.md": "Indexing in LlamaIndex involves parsing documents into nodes and storing them in an index structure. The `VectorStoreIndex` is a common choice, storing node embeddings in a vector database. Node parsers like `SentenceSplitter` or `SentenceWindowNodeParser` are crucial for preparing text for indexing. Efficient indexing is key for fast and accurate retrieval."
}

for filename, content in doc_contents.items():
    with open(os.path.join(corpus_dir, filename), "w") as f:
        f.write(content)

print(f"Corpus created in '{corpus_dir}' with {len(doc_contents)} documents.")

# Load the documents
documents = SimpleDirectoryReader(corpus_dir).load_data()
print(f"Loaded {len(documents)} documents.")

# --- Baseline Index and Query Engine --- 

# Create a basic VectorStoreIndex for comparison
baseline_index = VectorStoreIndex.from_documents(documents)
baseline_retriever = VectorIndexRetriever(
    index=baseline_index,
    similarity_top_k=2,
)
baseline_query_engine = RetrieverQueryEngine(
    retriever=baseline_retriever,
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.7) # Simple post-processing
    ]
)

print("Baseline index and query engine initialized.")


### Your Implementation

Now it's your turn! In the code cell below, implement at least two advanced retrieval techniques as described in the task requirements. We recommend focusing on **Sentence Window Retrieval** and **Hypothetical Document Embedding (HyDE)**.

For each technique:
1.  **Prepare the data**: If the technique requires specific node parsing or metadata, apply it.
2.  **Build the index**: Create a LlamaIndex `VectorStoreIndex` or other relevant index structure.
3.  **Construct the query engine**: Assemble the `RetrieverQueryEngine` with the appropriate retriever and any necessary post-processors or query transformations.
4.  **Perform qualitative evaluation**: Use the provided test queries (or create your own) to compare the retrieved context and generated answers from your advanced query engines against the `baseline_query_engine`.
5.  **Add comments**: Explain your code and your observations from the qualitative evaluation.


In [ ]:
# --- Reference Solution --- 

# --- Advanced Retrieval Technique 1: Sentence Window Retrieval --- 

print("\n--- Implementing Sentence Window Retrieval ---")

# 1. Prepare the data: Use SentenceWindowNodeParser
# This parser creates smaller 'sentence window' nodes for embedding and retrieval,
# but stores the full surrounding text in metadata for the LLM to use.
node_parser_sentence_window = SentenceWindowNodeParser(
    window_size=3, # Number of sentences to include on either side of the 'sentence window'
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

sentence_window_nodes = node_parser_sentence_window.get_nodes_from_documents(documents)

# For Sentence Window Retrieval, it's crucial to ensure that the original text
# is stored in the metadata so the post-processor can retrieve it.
# The SentenceWindowNodeParser handles this automatically by default.

# 2. Build the index
sentence_window_index = VectorStoreIndex(sentence_window_nodes)

# 3. Construct the query engine
sentence_window_retriever = VectorIndexRetriever(
    index=sentence_window_index,
    similarity_top_k=2,
)

# The MetadataReplacementPostProcessor replaces the small 'sentence window' with the
# full original text window before passing it to the LLM.
sentence_window_query_engine = RetrieverQueryEngine(
    retriever=sentence_window_retriever,
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window"),
        SimilarityPostprocessor(similarity_cutoff=0.7) # Optional: filter by similarity
    ]
)

print("Sentence Window Retrieval setup complete.")

# --- Advanced Retrieval Technique 2: Hypothetical Document Embedding (HyDE) --- 

print("\n--- Implementing Hypothetical Document Embedding (HyDE) ---")

# 1. Prepare the data: HyDE works on standard nodes, so we can reuse the original documents
# or create a new index from them.

# 2. Build the index: We'll use a standard VectorStoreIndex for HyDE.
# For demonstration, let's create a fresh index to avoid interference.
hyde_index = VectorStoreIndex.from_documents(documents)

# 3. Construct the query engine
# HyDE wraps a base retriever. The LLM first generates a hypothetical answer,
# which is then embedded and used for retrieval.
hyde_retriever = hyde_index.as_retriever(similarity_top_k=2)

hyde_query_engine = RetrieverQueryEngine(
    retriever=HypotheticalDocumentEmbedder(
        llm=Settings.llm, # Use the configured LLM to generate hypothetical answers
        base_retriever=hyde_retriever,
    ),
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.7)
    ]
)

print("HyDE setup complete.")

# --- Qualitative Evaluation --- 

print("\n--- Starting Qualitative Evaluation ---")

test_queries = [
    "What are the main components of LlamaIndex?",
    "How do advanced RAG patterns improve retrieval?",
    "Explain the process of indexing in LlamaIndex."
]

def print_response_and_sources(query_engine, query_name):
    print(f"\n--- Query Engine: {query_name} ---")
    response = query_engine.query(query)
    print(f"Query: {query}")
    print(f"Answer: {response.response}")
    print("Source Nodes (Top 2):")
    for i, node in enumerate(response.source_nodes[:2]):
        print(f"  Node {i+1} Score: {node.score:.2f}")
        # For Sentence Window, 'window' contains the expanded context, 'original_text' is the full doc
        # For others, 'text' is the node content.
        content_key = "window" if query_name == "Sentence Window Retrieval" else "text"
        print(f"  Content: {node.node.metadata.get(content_key, node.node.text)[:200]}...") # Show first 200 chars
        print(f"  Document ID: {node.node.id_}")

for query in test_queries:
    print(f"\n=====================================================")
    print(f"Evaluating Query: '{query}'")
    print(f"=====================================================")

    # Baseline Evaluation
    print_response_and_sources(baseline_query_engine, "Baseline Retrieval")

    # Sentence Window Retrieval Evaluation
    print_response_and_sources(sentence_window_query_engine, "Sentence Window Retrieval")

    # HyDE Evaluation
    print_response_and_sources(hyde_query_engine, "HyDE Retrieval")

    print("\n--- Comparison and Observations for this Query ---")
    print("**Baseline Retrieval:** Often retrieves broader chunks of text. May include irrelevant sentences alongside relevant ones, potentially diluting the context for the LLM. The answer quality depends heavily on the initial embedding match.")
    print("**Sentence Window Retrieval:** Typically retrieves a very precise sentence or small window for initial embedding, then expands it to a larger, more coherent context for the LLM. This often leads to more focused and relevant context, improving the LLM's ability to synthesize an accurate answer. The retrieved context should feel more 'to the point' while still providing sufficient surrounding information.")
    print("**HyDE Retrieval:** The LLM first generates a hypothetical answer based on the query. This hypothetical answer is then embedded and used to find documents. This can be very effective for queries where the original query embedding might not perfectly match the document content, but a 'hypothetical' answer's embedding would. It helps bridge the semantic gap, potentially leading to retrieval of documents that are conceptually relevant even if they don't contain exact keywords. The retrieved context might be broader but semantically richer.")
    print("-----------------------------------------------------")

print("\n--- Qualitative Evaluation Complete ---")
print("Review the outputs above to understand the differences in retrieved context and answers for each technique.")

# Clean up dummy corpus directory
import shutil
shutil.rmtree(corpus_dir)
print(f"Cleaned up corpus directory: {corpus_dir}")
